# Feat. Engineering

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
project_root = (
    Path.cwd().parent
    if Path.cwd().name == 'notebooks'
    else Path.cwd()
)

input_path = (
    project_root
    / 'data'
    / 'interim'
    / 'ca1_development_joined.parquet'
)

output_path = (
    project_root
    / 'data'
    / 'processed'
    / 'ca1_model_features.parquet'
)

print(f'Input: {input_path}')
print(f'Output: {output_path}')

Input: c:\Users\ryana\OneDrive - MMU\Documents\MSc Project\retail-demand-forecasting\data\interim\ca1_development_joined.parquet
Output: c:\Users\ryana\OneDrive - MMU\Documents\MSc Project\retail-demand-forecasting\data\processed\ca1_model_features.parquet


In [3]:
# Loading the main dataset
model_df = pd.read_parquet(input_path)

print(f'Rows: {len(model_df):,}')
print(f'Items: {model_df["item_id"].nunique():,}')

model_df.head()

Rows: 209,628
Items: 108


,id,item_id,dept_id,cat_id,store_id,state_id,d,sales,date,wm_yr_wk,weekday,wday,month,year,event_name_1,event_type_1,event_name_2,event_type_2,sell_price
0,FOODS_1_010_CA_1_evaluation,FOODS_1_010,FOODS_1,FOODS,CA_1,CA,d_1,0,2011-01-29,11101,Saturday,1,1,2011,NaN,NaN,NaN,NaN,NaN
1,FOODS_1_010_CA_1_evaluation,FOODS_1_010,FOODS_1,FOODS,CA_1,CA,d_2,0,2011-01-30,11101,Sunday,2,1,2011,NaN,NaN,NaN,NaN,NaN
2,FOODS_1_010_CA_1_evaluation,FOODS_1_010,FOODS_1,FOODS,CA_1,CA,d_3,0,2011-01-31,11101,Monday,3,1,2011,NaN,NaN,NaN,NaN,NaN
3,FOODS_1_010_CA_1_evaluation,FOODS_1_010,FOODS_1,FOODS,CA_1,CA,d_4,0,2011-02-01,11101,Tuesday,4,2,2011,NaN,NaN,NaN,NaN,NaN
4,FOODS_1_010_CA_1_evaluation,FOODS_1_010,FOODS_1,FOODS,CA_1,CA,d_5,0,2011-02-02,11101,Wednesday,5,2,2011,NaN,NaN,NaN,NaN,NaN


In [4]:
# Checking the required features exists
required_columns = [
    'item_id',
    'dept_id',
    'cat_id',
    'store_id',
    'date',
    'sales',
    'weekday',
    'wday',
    'month',
    'event_name_1',
    'event_name_2',
    'sell_price'
]

missing_columns = [
    column
    for column in required_columns
    if column not in model_df.columns
]

assert not missing_columns, (f'Missing required columns: {missing_columns}')

In [5]:
# Sorting the observations correctly
model_df['date'] = pd.to_datetime(model_df['date'])

model_df = model_df.sort_values(['store_id', 'item_id', 'date']).reset_index(drop=True)

In [6]:
# Checking any duplicates
duplicate_count = model_df.duplicated(
    subset=[
        'store_id',
        'item_id',
        'date'
    ]
).sum()

print(f'Duplicate item-date rows: {duplicate_count:,}')

assert duplicate_count == 0

Duplicate item-date rows: 0


In [7]:
# Defining the time-series key
series_keys = ['store_id', 'item_id']

Lag is a specific sale periods in the past to predict sales for a current specific date

In [8]:
# Creating lag features
lag_periods = [1, 7, 14, 28]

for lag in lag_periods:
    model_df[f'lag_{lag}'] = (
        model_df
        .groupby(series_keys)['sales']
        .shift(lag)
    )

The shift must happen in each time series before calculating to prevent leakage

In [9]:
# Creating leakage-self rolling features
model_df['rolling_mean_7'] = (
    model_df
    .groupby(series_keys)['sales']
    .transform(
        lambda series:
        series.shift(1)
        .rolling(
            window=7,
            min_periods=7
        )
        .mean()
    )
)

In [10]:
model_df['rolling_mean_28'] = (
    model_df
    .groupby(series_keys)['sales']
    .transform(
        lambda series:
        series.shift(1)
        .rolling(
            window=28,
            min_periods=28
        )
        .mean()
    )
)

In [11]:
model_df['rolling_std_7'] = (
    model_df
    .groupby(series_keys)['sales']
    .transform(
        lambda series:
        series.shift(1)
        .rolling(
            window=7,
            min_periods=7
        )
        .std()
    )
)

This is to prevent, when extracting rolling mean for day X, only includes sales from day X-7 to day X-1 excluding the predicted day (X)

In [12]:
# Creating event indicator
event_columns = [
    'event_name_1',
    'event_name_2'
]

model_df['event_indicator'] = (
    model_df[event_columns]
    .notna()
    .any(axis=1)
    .astype('int8')
)

In [13]:
# Checking the resu;ts
model_df['event_indicator'].value_counts()

event_indicator
0    192564
1     17064
Name: count, dtype: int64

0 indicates no event, 1 indicates at least one event was recorded

In [14]:
# Create price change
previous_price = (
    model_df
    .groupby(series_keys)['sell_price']
    .shift(1)
)

model_df['price_change'] = (
    (
        model_df['sell_price']
        - previous_price
    )
    / previous_price
)

In [15]:
# Preventing division by zero 
model_df['price_change'] = (
    model_df['price_change']
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
)

Interpretation; 0.00: no price change; -0.10: decrease by 10%; 0.10: increase by 10%

In [16]:
# Keeping only the useful columns
feature_columns = [
    # Observation identifiers
    'id',
    'store_id',
    'state_id',
    'item_id',
    'dept_id',
    'cat_id',
    'd',
    'date',
    'wm_yr_wk',

    # Prediction target
    'sales',

    # Calendar predictors
    'weekday',
    'wday',
    'month',
    'event_indicator',

    # Price predictors
    'sell_price',
    'price_change',

    # Demand-history predictors
    'lag_1',
    'lag_7',
    'lag_14',
    'lag_28',
    'rolling_mean_7',
    'rolling_mean_28',
    'rolling_std_7'
]

feature_df = model_df[feature_columns].copy()

feature_df.head()

,id,store_id,state_id,item_id,dept_id,cat_id,d,date,wm_yr_wk,sales,...,event_indicator,sell_price,price_change,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28,rolling_std_7
0,FOODS_1_010_CA_1_evaluation,CA_1,CA,FOODS_1_010,FOODS_1,FOODS,d_1,2011-01-29,11101,0,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,FOODS_1_010_CA_1_evaluation,CA_1,CA,FOODS_1_010,FOODS_1,FOODS,d_2,2011-01-30,11101,0,...,0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,FOODS_1_010_CA_1_evaluation,CA_1,CA,FOODS_1_010,FOODS_1,FOODS,d_3,2011-01-31,11101,0,...,0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
3,FOODS_1_010_CA_1_evaluation,CA_1,CA,FOODS_1_010,FOODS_1,FOODS,d_4,2011-02-01,11101,0,...,0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN
4,FOODS_1_010_CA_1_evaluation,CA_1,CA,FOODS_1_010,FOODS_1,FOODS,d_5,2011-02-02,11101,0,...,0,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN


item_id, dept_id, cat_id, and weekday remain categorical identifiers. These will be encoded during model processing instead of in this notebook

In [17]:
# Inspecting the missing feature values
engineered_features = [
    'lag_1',
    'lag_7',
    'lag_14',
    'lag_28',
    'rolling_mean_7',
    'rolling_mean_28',
    'rolling_std_7',
    'sell_price',
    'price_change'
]

feature_df[engineered_features].isna().sum()

lag_1                108
lag_7                756
lag_14              1512
lag_28              3024
rolling_mean_7       756
rolling_mean_28     3024
rolling_std_7        756
sell_price         41594
price_change       41702
dtype: int64

Lag and Rolling missing values are expected

In [18]:
missing_price_sales = feature_df.loc[feature_df['sell_price'].isna(), 'sales']

print(
    f'Rows with missing prices: '
    f'{len(missing_price_sales):,}'
)

print(
    f'Rows with missing prices and positive sales: '
    f'{missing_price_sales.gt(0).sum():,}'
)

print(
    f'Total sales where price is missing: '
    f'{missing_price_sales.sum():,}'
)

Rows with missing prices: 41,594
Rows with missing prices and positive sales: 0
Total sales where price is missing: 0


In [19]:
# checking for leakage
sample_item = feature_df['item_id'].iloc[0]

feature_df.loc[
    feature_df['item_id'] == sample_item,
    [
        'date',
        'sales',
        'lag_1',
        'lag_7',
        'rolling_mean_7'
    ]
].head(10)

,date,sales,lag_1,lag_7,rolling_mean_7
0,2011-01-29,0,NaN,NaN,NaN
1,2011-01-30,0,0.0,NaN,NaN
2,2011-01-31,0,0.0,NaN,NaN
3,2011-02-01,0,0.0,NaN,NaN
4,2011-02-02,0,0.0,NaN,NaN
5,2011-02-03,0,0.0,NaN,NaN
6,2011-02-04,0,0.0,NaN,NaN
7,2011-02-05,0,0.0,0.0,0.0
8,2011-02-06,0,0.0,0.0,0.0
9,2011-02-07,0,0.0,0.0,0.0


In [20]:
# Automatically verify lag_1
expected_lag_1 = (
    feature_df
    .groupby(series_keys)['sales']
    .shift(1)
)

assert np.allclose(
    feature_df['lag_1'],
    expected_lag_1,
    equal_nan=True
)

print('lag_1 correctly contains previous-day sales.')

lag_1 correctly contains previous-day sales.


In [21]:
# Verify the 7-day rolling mean
expected_rolling_mean_7 = (
    feature_df
    .groupby(series_keys)['sales']
    .transform(
        lambda series:
        series.shift(1)
        .rolling(
            window=7,
            min_periods=7
        )
        .mean()
    )
)

assert np.allclose(
    feature_df['rolling_mean_7'],
    expected_rolling_mean_7,
    equal_nan=True
)

In [22]:
# Remove rows without sufficient sales history
history_features = [
    'lag_1',
    'lag_7',
    'lag_14',
    'lag_28',
    'rolling_mean_7',
    'rolling_mean_28',
    'rolling_std_7'
]

rows_before = len(feature_df)

feature_df = feature_df.dropna(subset=history_features).reset_index(drop=True)

rows_removed = (rows_before - len(feature_df))

print(f'Rows before: {rows_before:,}')
print(f'Rows removed: {rows_removed:,}')
print(f'Rows remaining: {len(feature_df):,}')

Rows before: 209,628
Rows removed: 3,024
Rows remaining: 206,604


In [25]:
first_price_dates = (feature_df.loc[feature_df['sell_price'].notna()]
    .groupby(series_keys)['date']
    .min()
    .rename('first_price_date')
    .reset_index()
)

price_check_df = feature_df.merge(
    first_price_dates,
    on=series_keys,
    how='left'
)

missing_after_first_price = price_check_df.loc[price_check_df['sell_price'].isna()
    & (price_check_df['date'] >= price_check_df['first_price_date'])
]

print(
    f'Missing prices after first recorded price: '
    f'{len(missing_after_first_price):,}'
)

Missing prices after first recorded price: 0


In [26]:
feature_df = price_check_df.loc[
    price_check_df['date'] >= price_check_df['first_price_date']
].drop(columns='first_price_date').reset_index(drop=True)

In [27]:
feature_df['price_change'] = (feature_df['price_change'].fillna(0))

In [28]:
print(feature_df[
        ['sell_price', 'price_change']
    ].isna().sum())

print(f'Final rows: {len(feature_df):,}')

sell_price      0
price_change    0
dtype: int64
Final rows: 166,886


missing sell_price and price_change are removed later in the pipeline to maintain consistency

In [29]:
# final checks
print(f'Dataset shape: {feature_df.shape}')

print(
    f'Items represented: '
    f'{feature_df["item_id"].nunique():,}'
)

print(
    f'Date range: '
    f'{feature_df["date"].min().date()} '
    f'to '
    f'{feature_df["date"].max().date()}'
)

print(
    f'Duplicate observations: '
    f'{feature_df.duplicated(subset=series_keys + ["date"]).sum():,}'
)

Dataset shape: (166886, 23)
Items represented: 108
Date range: 2011-02-26 to 2016-05-22
Duplicate observations: 0


In [30]:
history_features = [
    'lag_1',
    'lag_7',
    'lag_14',
    'lag_28',
    'rolling_mean_7',
    'rolling_mean_28',
    'rolling_std_7'
]

# Remove observations without sufficient demand history
feature_df = feature_df.dropna(subset=history_features).copy()

print(
    f'Rows after history filtering: '
    f'{len(feature_df):,}'
)

# Find the first recorded price for each product
first_price_dates = (
    feature_df.loc[
        feature_df['sell_price'].notna()
    ]
    .groupby(series_keys)['date']
    .min()
    .rename('first_price_date')
    .reset_index()
)

feature_df = feature_df.merge(
    first_price_dates,
    on=series_keys,
    how='left'
)

# Verify that missing prices occur only before availability
missing_after_first_price = feature_df.loc[
    feature_df['sell_price'].isna()
    & (feature_df['date'] >= feature_df['first_price_date'])
]

print(
    f'Missing prices after first recorded price: '
    f'{len(missing_after_first_price):,}'
)

assert len(missing_after_first_price) == 0

# Remove pre-availability observations
feature_df = feature_df.loc[
    feature_df['date']
    >= feature_df['first_price_date']
].drop(columns='first_price_date').reset_index(drop=True)

# No earlier price exists for the first available observation
feature_df['price_change'] = (feature_df['price_change'].fillna(0))

Rows after history filtering: 166,886
Missing prices after first recorded price: 0


In [31]:
columns_to_check = (history_features + ['sell_price', 'price_change'])

print(f'Final rows: {len(feature_df):,}')

print(
    f'Date range: '
    f'{feature_df["date"].min().date()} '
    f'to {feature_df["date"].max().date()}'
)

print(
    feature_df[columns_to_check].isna().sum()
)

assert (
    feature_df[columns_to_check].isna().sum().sum() == 0
)

Final rows: 166,886
Date range: 2011-02-26 to 2016-05-22
lag_1              0
lag_7              0
lag_14             0
lag_28             0
rolling_mean_7     0
rolling_mean_28    0
rolling_std_7      0
sell_price         0
price_change       0
dtype: int64


In [32]:
feature_df.to_parquet(output_path, index=False)

saved_check = pd.read_parquet(output_path)

print(f'Saved rows: {len(saved_check):,}')
print(
    f'Saved first date: '
    f'{saved_check["date"].min().date()}'
)

Saved rows: 166,886
Saved first date: 2011-02-26


### Finished Structure (inspection)

In [33]:
# Features
feature_df.head()

,id,store_id,state_id,item_id,dept_id,cat_id,d,date,wm_yr_wk,sales,...,event_indicator,sell_price,price_change,lag_1,lag_7,lag_14,lag_28,rolling_mean_7,rolling_mean_28,rolling_std_7
0,FOODS_1_010_CA_1_evaluation,CA_1,CA,FOODS_1_010,FOODS_1,FOODS,d_484,2012-05-26,11218,0,...,0,5.09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,FOODS_1_010_CA_1_evaluation,CA_1,CA,FOODS_1_010,FOODS_1,FOODS,d_485,2012-05-27,11218,0,...,0,5.09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,FOODS_1_010_CA_1_evaluation,CA_1,CA,FOODS_1_010,FOODS_1,FOODS,d_486,2012-05-28,11218,0,...,1,5.09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,FOODS_1_010_CA_1_evaluation,CA_1,CA,FOODS_1_010,FOODS_1,FOODS,d_487,2012-05-29,11218,0,...,0,5.09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,FOODS_1_010_CA_1_evaluation,CA_1,CA,FOODS_1_010,FOODS_1,FOODS,d_488,2012-05-30,11218,0,...,0,5.09,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
# Types
feature_df.dtypes

id                            str
store_id                      str
state_id                      str
item_id                       str
dept_id                       str
cat_id                        str
d                             str
date               datetime64[us]
wm_yr_wk                    int64
sales                       int64
weekday                       str
wday                        int64
month                       int64
event_indicator              int8
sell_price                float64
price_change              float64
lag_1                     float64
lag_7                     float64
lag_14                    float64
lag_28                    float64
rolling_mean_7            float64
rolling_mean_28           float64
rolling_std_7             float64
dtype: object

In [36]:
# Saving
output_path.parent.mkdir(
    parents=True, exist_ok=True
)

feature_df.to_parquet(
    output_path, index=False
)

print(f'Saved to: {output_path}')

Saved to: c:\Users\ryana\OneDrive - MMU\Documents\MSc Project\retail-demand-forecasting\data\processed\ca1_model_features.parquet
